# v26 champion 실행/재현 노트북

공식 DACON 제출 이력(제출 ID 기준, 2026-08-18 직접 대조)으로 확인한 실제 champion은 `submit_v26.zip`이다 — Public **1157.9736407889**, 제출 ID **51773**, 제출 2026-08-17 05:08:58.

> **문서 정정 안내**: 저장소의 일부 과거 문서(`README.md`의 현재 현황 표, `reports/target1170_followup_20260817.md`, `reports/submissions.csv`의 v26/v27/v28 행)는 `submit_v27.zip`을 champion(1157.9736407889, 제출 ID `1535195`)으로 잘못 기록하고 있다. `1535195`는 DACON이 실제로 발급하는 5자리 제출 ID 형식(`51773`, `51775`, `51783` ...)과 다르며, 공식 제출 이력을 다시 대조한 결과 다음이 확인됐다.

| 파일 | 실제 Public 점수 | 제출 ID | 제출 일시 |
|---|---:|---|---|
| `submit_v25.zip` | 1155.8293405409 | 51722 | 2026-08-17 03:37:45 |
| `submit_v26.zip` | **1157.9736407889 (champion)** | 51773 | 2026-08-17 05:08:58 |
| `submit_v27.zip` | 1156.6153781694 | 51775 | 2026-08-17 05:14:03 |
| `submit_v28.zip` | 1156.9983034655 | 51783 | 2026-08-17 05:54:23 |

이 노트북은 위 정정된 사실을 기준으로 champion(v26)을 실행·검증하고 동일 recipe로 재빌드한다. 계보는 `submit_v22.zip` → (`src/package_v25_postbreak_anchor.py`) → `submit_v25.zip` → (`src/package_v26_anchor_weight_probe.py`, 기본값 `--probe-eta 0.15`) → `submit_v26.zip` 이다. v25/v22의 점수와 해시는 기존 문서와 실제 DACON 값이 일치해 정정 대상이 아니다.

Python 모듈(`src/`)이 계산의 단일 원본이다. 이 노트북은 그 모듈을 순서대로 호출하고 결과를 확인하는 얇은 실행 화면이며 학습·검증 로직을 셀 안에 복사하지 않는다. 처음 열면 모든 스위치가 안전한 기본값(`DRY_RUN=True` 등)으로 되어 있으므로 위에서 아래로 그냥 실행해 명령과 경로만 먼저 확인해도 된다.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import os
import subprocess
import sys


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'reports').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f'project: {PROJECT_ROOT}')
print(f'python : {sys.executable}')
print(f'version: {sys.version.split()[0]}')

## 실행 스위치

- `DRY_RUN`: `True`이면 실행할 명령만 출력한다.
- `RUN_TESTS`: 데이터 비의존 테스트와 저장소 audit을 실행한다.
- `RUN_OFFICIAL_VALIDATION`: 실제 `submit_v26.zip`을 풀어 245,789행 전체를 추론하고 공식 게이트를 재현한다. `data/train.csv`, `data/test.csv`, `submit_v22.zip`, `submit_v26.zip`이 필요하다.
- `RUN_REBUILD`: `submit_v25.zip`에서 `--probe-eta 0.15`로 v26을 처음부터 다시 패키징한다. `submit_v25.zip`이 필요하다.
- `RUN_REBUILD_VALIDATION`: 방금 재빌드한 zip을 v22 기준으로 다시 검증한다.

원본 데이터와 제출 ZIP은 Git에 없다. 각 팀원이 DACON에서 데이터를 받고, 팀이 합의한 private artifact 경로에서 ZIP을 받아 프로젝트 루트에 둔다.

In [ ]:
DRY_RUN = True
RUN_TESTS = True
RUN_OFFICIAL_VALIDATION = False
RUN_REBUILD = False
RUN_REBUILD_VALIDATION = False

V26_PUBLIC = 1157.9736407889
V26_SUBMISSION_ID = '51773'
TARGET_PUBLIC = 1200.0

KNOWN_SHA256 = {
    'submit_v22.zip': 'FB12FC236B82E6D72D1A870F3F8C52CBD8C5D79D31E3F0E98193E0F648BB47B7',
    'submit_v25.zip': '60CF13B28AF01B24A49E2B03E13161F2E04A1C09647D8DA2CFD95407ED9335B1',
    'submit_v26.zip': '8BE26E156A91C1FA5E9989EEE5043DB5992A960DF35A53D8A9BFEF7E6825D096',
}

print(f'gap to 1170: {1170.0 - V26_PUBLIC:.10f}')
print(f'gap to target {TARGET_PUBLIC:.0f}: {TARGET_PUBLIC - V26_PUBLIC:.10f}')

In [ ]:
def command_text(command):
    return subprocess.list2cmdline([str(part) for part in command])


def run_command(command, *, enabled, label):
    command = [str(part) for part in command]
    print(f'[{label}] {command_text(command)}')
    if DRY_RUN:
        print('  DRY_RUN: 실행하지 않음')
        return None
    if not enabled:
        print('  스위치가 꺼져 있어 실행하지 않음')
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)


def run_module(module, *args, enabled, label):
    return run_command(
        [sys.executable, '-m', module, *args], enabled=enabled, label=label
    )


def sha256_of(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest().upper()

## 1. 입력 파일 확인

데이터와 ZIP은 Git 대상이 아니다. 아래에서 READY/MISSING만 먼저 확인한다.

In [ ]:
required_paths = {
    'train data': Path('data/train.csv'),
    'test data': Path('data/test.csv'),
    'v22 (검증 기준선)': Path('submit_v22.zip'),
    'v25 (v26 직접 부모)': Path('submit_v25.zip'),
    'v26 champion (공식)': Path('submit_v26.zip'),
}
for name, path in required_paths.items():
    state = 'READY' if path.exists() else 'MISSING'
    print(f'{state:7}  {name:20}  {path}')

## 2. SHA-256 대조

실행 전에 받은 파일이 팀이 기록한 파일과 정확히 같은지 먼저 확인한다. 데이터 없이도 실행할 수 있다.

In [ ]:
for name, expected in KNOWN_SHA256.items():
    path = Path(name)
    if not path.exists():
        label = 'SKIP'
        print(f'{label:9} {name}  (파일 없음)')
        continue
    actual = sha256_of(path)
    verdict = 'MATCH' if actual == expected else 'MISMATCH'
    print(f'{verdict:9} {name}  {actual}')

## 3. 데이터 비의존 테스트

저장소 audit과 데이터 없이 실행 가능한 pytest 스위트를 먼저 돌린다. `DRY_RUN = False`, `RUN_TESTS = True`일 때만 실행된다.

In [ ]:
run_command(
    [sys.executable, 'scripts/audit_repository.py', '--include-untracked'],
    enabled=RUN_TESTS, label='repository audit',
)
run_module(
    'pytest', '-q',
    'tests/test_metrics.py',
    'tests/test_local_scorecard.py',
    'tests/test_robust_local_evaluation.py',
    'tests/test_v25_postbreak_anchor.py',
    'tests/test_v26_anchor_weight_probe.py',
    'tests/test_experiment_notebook.py',
    enabled=RUN_TESTS, label='data-free tests',
)

## 4. champion(v26) 실제 실행

`submit_v26.zip`을 실제로 풀어 전체 245,789행을 추론하고, `submit_v22.zip`을 기준으로 lineage·수식·batch invariance·runtime·memory 게이트를 재현한다. 공식 결과는 [`../reports/v26_validation.json`](../reports/v26_validation.json)(runtime 56.240초, peak 1410.7MB, 모든 게이트 통과)에 있으므로 이 셀의 출력과 비교해서 확인한다. 결과 리포트는 `reports/v26_notebook_validation.md`에 저장되며 기존 공식 리포트는 덮어쓰지 않는다.

In [ ]:
run_module(
    'src.validate_v25_postbreak_anchor',
    '--project', '.',
    '--candidate', 'submit_v26.zip',
    '--parent', 'submit_v22.zip',
    '--report', 'reports/v26_notebook_validation.md',
    enabled=RUN_OFFICIAL_VALIDATION,
    label='run champion v26 end-to-end (matches reports/v26_validation.json)',
)

## 5. 동일 recipe로 v26 재빌드

`submit_v25.zip`을 부모로 `--probe-eta 0.15`(스크립트 기본값)를 적용해 `submit_v26_rebuild.zip`을 새로 만든다. 이미 있는 `submit_v26.zip`은 절대 덮어쓰지 않는다(스크립트가 기존 출력 파일이 있으면 예외를 던진다).

In [ ]:
REBUILD_ZIP = 'submit_v26_rebuild.zip'
run_module(
    'src.package_v26_anchor_weight_probe',
    '--project', '.',
    '--parent', 'submit_v25.zip',
    '--output', REBUILD_ZIP,
    '--probe-eta', '0.15',
    enabled=RUN_REBUILD,
    label='rebuild v26 (v25 + 15% R_ANCHOR probe, script default)',
)

In [ ]:
rebuild_path = Path(REBUILD_ZIP)
if rebuild_path.exists():
    actual = sha256_of(rebuild_path)
    expected = KNOWN_SHA256['submit_v26.zip']
    verdict = 'MATCH' if actual == expected else 'MISMATCH'
    print(f'{verdict:9} {REBUILD_ZIP}  {actual}')
else:
    label = 'SKIP'
    print(f'{label:9} {REBUILD_ZIP}  (아직 생성되지 않음)')

## 6. 재빌드 검증

재빌드한 ZIP도 v22 기준으로 같은 게이트를 통과하는지 확인한다.

In [ ]:
run_module(
    'src.validate_v25_postbreak_anchor',
    '--project', '.',
    '--candidate', REBUILD_ZIP,
    '--parent', 'submit_v22.zip',
    '--report', 'reports/v26_rebuild_notebook_validation.md',
    enabled=RUN_REBUILD_VALIDATION,
    label='validate rebuild vs v22 lineage',
)

## 7. 최근 제출 이력

`reports/submissions.csv`에서 최근 제출과 판정을 확인한다. v26/v27/v28 행의 `public_score`는 위 표에 맞춰 이 브랜치에서 정정했다(정정 근거는 각 행의 `notes`에 함께 기록했다).

In [ ]:
with Path('reports/submissions.csv').open(encoding='utf-8-sig', newline='') as stream:
    submission_rows = list(csv.DictReader(stream))

for row in submission_rows[-5:]:
    print({key: row.get(key) for key in ('name', 'public_score', 'submitted_at', 'notes')})

## 다음 단계

1. 위 셀에서 READY/MATCH가 모두 확인된 뒤에만 `DRY_RUN = False`로 바꾼다.
2. `RUN_OFFICIAL_VALIDATION`으로 먼저 실제 champion(v26)을 돌려보고 `reports/v26_validation.json`과 비교한다.
3. 재현을 확인하려면 `RUN_REBUILD`, `RUN_REBUILD_VALIDATION`을 차례로 켠다.
4. 새 후보는 이 노트북을 수정하지 말고 `docs/EXPERIMENT_WORKFLOW.md`의 절차를 따라 별도 이름으로 만들고, champion(v26)을 부모로 삼는다.
5. `reports/target1170_followup_20260817.md`, `README.md`의 과거 champion 서술(v27 기준)은 이 정정 이전에 작성된 것이므로 새 의사결정의 근거로 그대로 인용하지 않는다.
6. champion을 넘는 새 제출이 나오면 `README.md`, `reports/submissions.csv`, 이 노트북의 `KNOWN_SHA256`을 함께 갱신한다.